# Tutorial

## 00 - globals

In [ ]:
import os
from stamp.config import adata_dir

In [ ]:
slides = {
    1: {
        "exprmat": "exprMat_file.csv.gz",
        "metadata": "metadata_file.csv.gz",
        "fov_positions": "fov_positions_file.csv.gz",
    }
}
# optional filepath prefix
data_dir = "data"

In [ ]:
for slide, d in slides.items():
    if not isinstance(slide, int):
        raise ValueError(f"Keys for dictionary `slides` must be integer values, not {type(slide)=} (found {slide=})")
    for k, v in d.items():
        f = os.path.join(data_dir, v)
        if not os.path.exists(f):
            raise FileNotFoundError(f)

In [ ]:
os.makedirs(adata_dir, exist_ok=True)

## 01 - Read

In [ ]:
import pandas as pd
import scanpy as sc
from stamp.config import sample_md_columns, adata_dir
from stamp.read import read_cosmx

In [ ]:
sample_df = pd.read_table("data/sample2fov.csv", sep=",")
sample_df.sort_values(["slide", "fov_start"], inplace=True)
sample_df

In [ ]:
for col in sample_md_columns:
    if col not in sample_df.columns:
        raise ValueError(f"column={col} not found in sample_df")
for slide in slides:
    if slide not in set(sample_df["slide"]):
        raise ValueError(f"{slide=} not found in sample_df['slide']")

In [ ]:
adata_file = os.path.join(adata_dir, "raw_data.h5ad")
read_cosmx(
    slides,
    sample_df,
    adata_file,
    # sample_df_columns=["Donor", "Treatment", "Timepoint"],
    # metadata_df_columns=[
    #     'nCount_RNA', 'nFeature_RNA', "nCount_negprobes", "nCount_falsecode", "Area.um2", "qcFlagsFOV"
    # ],
    # nrows=100,
    data_dir=data_dir,
    overwrite=False,
    verbose=True,
)

In [ ]:
adata = sc.read_h5ad(adata_file)
adata.obs

## 02 - QC

In [ ]:
import matplotlib.pyplot as plt

from stamp.qc import slide_qc_data, slide_qc_plots

In [ ]:
slide_qc_data(adata, slides, data_dir)
adata.uns["fov_metadata"]

In [ ]:
fig_axs_list = slide_qc_plots(adata, columns=["slide", "x", "y", "nCounts", "nCell"])
plt.show()

In [ ]:
from stamp.qc import plot_avg_per_pixel

In [ ]:
idx = adata.obs[(adata.obs["slide-fov"] == "1-15")].index
fig, axs = plot_avg_per_pixel(adata[idx, :], column='nCount_RNA', log1p=False, fill_cell_area=True)
plt.show()

In [ ]:
fig, axs = plot_avg_per_pixel(adata, column='nCount_RNA', log1p=False, fill_cell_area=True)
plt.show()

In [ ]:
from stamp.qc import violin, gene_qc, plot_distribution

In [ ]:
fig, axs = violin(adata, ["nCount_RNA", "nFeature_RNA", "Area.um2"])
plt.show()
fig, axs = violin(adata, ["nCount_negprobes", "nCount_falsecode"], log_scale=(False, False))
plt.show()

In [ ]:
gene_qc(adata)
adata.var

In [ ]:
fig, ax = plot_distribution(adata, "nCell", axis=1)
plt.show()

## 03 - Filtering

In [ ]:
import seaborn as sns

from stamp.filter import genes_signal_to_noise, filter_genes, filter_cells

In [ ]:
genes_signal_to_noise(adata, 1)

# randomly assign genes above noise threshold for test data
random_genes = adata.var.sample(n = round(len(adata.var) * 0.8), random_state=42).index
adata.var["above_noise"] = adata.var.index.isin(random_genes)

adata.var["above_noise"].value_counts()

In [ ]:
"""add a custom filter"""
# add a column to adata.obs with the filter as boolean
threshold = adata.obs["dist2edge_px"].quantile(0.05)
adata.obs["above_distance"] = adata.obs["dist2edge_px"] >= threshold

# plot the threshold
fig, ax = plt.subplots()
sns.histplot(adata.obs["dist2edge_px"], bins=50, ax=ax)
ax.axvline(threshold, color="red")
plt.show()

In [ ]:
adata = filter_genes(adata, ncell_min=10)
adata = filter_cells(adata, filter_columns=["above_distance"])

In [ ]:
from stamp.qc import plot_ncell_per_condition, plot_value_distribution

In [ ]:
fig, ax = plot_ncell_per_condition(
    adata,
    columns = ["Treatment", "Timepoint"],
    offset_between_conditions = 1,
)
plt.show()
fig, ax = plot_ncell_per_condition(
    adata,
    columns = ["Treatment", "Timepoint", "Donor"],
    offset_between_conditions = [5, 1],
    text_kwargs={"size": 4},
)
plt.show()

In [ ]:
fig, ax = plot_value_distribution(adata)
plt.show()

# 04 - Binarize

In [ ]:
from stamp.process import binarize

In [ ]:
binarize(adata)
adata_file = os.path.join(adata_dir, "filtered.h5ad")
adata.write_h5ad(adata_file)

In [ ]:
adata.layers

In [ ]:
fig, ax = plot_value_distribution(adata, min_quantile=0.00, max_quantile=1.00)
plt.show()

In [ ]:
adata.var["nCell_postfilter"] = adata.layers["binary"].count_nonzero(axis=0)
adata.var["pctCell_postfilter"] = 100 * adata.var["nCell_postfilter"] / adata.n_obs
adata.var

In [ ]:
adata.obs["nFeature_RNA_postfilter"] = adata.layers["binary"].count_nonzero(axis=1)
adata.obs["nCount_RNA_postfilter"] = adata.layers["counts"].count_nonzero(axis=1)
adata.obs

# 05 - Dimensionality reduction

In [ ]:
from stamp.pca import pca, scree_plot, plot_pca

In [ ]:
pca(adata)
fig, ax = scree_plot(adata)
plt.show()

In [ ]:
plot_pca(
    adata, 
    color=["Treatment", "Timepoint"],
)

In [ ]:
sc.pp.neighbors(
    adata,
    use_rep="X_svd",
    key_added="neighbors_svd",
)

In [ ]:
sc.tl.umap(
    adata,
    neighbors_key="neighbors_svd",
    key_added="umap_svd",
)

In [ ]:
sc.pl.embedding(
    adata,
    basis="umap_svd",
    alpha=0.5,
)

# 06 - Clustering

In [ ]:
import numpy as np

In [ ]:
knn = adata.obsp["neighbors_svd_connectivities"].copy()
knn.data = np.ones_like(knn.data)
knn.setdiag(1)
deg = np.array(knn.sum(axis=1)).ravel()
knn = knn.multiply(1 / deg[:, None])
adata.layers["KNN_binary_mean"] = knn.dot(adata.layers["binary"]).toarray().astype(np.float32)

In [ ]:
markerdict = {
    "T_cells": [
        "CD3D", "CD3E", "TRBC1",
        "CD4", "IL7R", "CD8A", "CD8B",
    ],
    "B_cells": [
        "MS4A1", "CD79A", "CD79B", "CD37"
    ],
    "NK_cells": [
        "NKG7", "GNLY", "KLRD1", "FCGR3A"
    ],
    "Monocytes_Macrophages": [
        "LYZ", "LST1", "S100A8", "S100A9",
        "CTSD", "CST3", "LGALS3",
        "CD14", "CD68"
    ],
    "Dendritic_cells": [
        "FCER1A", "CST3", "CLEC10A", "CD1C"
    ]
}

for ctype, markers in markerdict.items():
    for marker in set(markers):
        if marker not in adata.var_names:
            print(f"{ctype} marker gene '{marker}' is not present in adata.var_names (removing)")
            markerdict[ctype].remove(marker)

In [ ]:
for ctype, markers in markerdict.items():
    if len(markers) == 0:
        continue
        
    print(ctype)
    sc.pl.embedding(
        adata,
        basis="umap_svd",
        color=markers,
        layer="KNN_binary_mean",
        ncols=3,
        alpha=0.5,
    )

In [ ]:
sc.tl.leiden(
    adata,
    resolution=0.6,
    neighbors_key="neighbors_svd",
    key_added="leiden_res_0.60",
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

In [ ]:
sc.pl.embedding(
    adata,
    basis="umap_svd",
    color="leiden_res_0.60",
    alpha=0.5,
)

In [ ]:
mp = sc.pl.MatrixPlot(
    adata,
    markerdict,
    groupby="leiden_res_0.60",
    layer="KNN_binary_mean",
    var_group_rotation=0,
)
ax = mp.get_axes()["mainplot_ax"]
_ = plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

In [ ]:
sc.pl.heatmap(
    adata, 
    markerdict,
    groupby="leiden_res_0.60",
    layer="KNN_binary_mean",
    var_group_rotation=0,
)

In [ ]:
fig, ax = plot_value_distribution(adata, layer = "KNN_binary_mean")
plt.show()